# 第 41 章：Capstone 报告、展示与最终签字

这个 notebook 用一个极小的 final delivery review 表，对比“只看 slide polish”的 naive baseline 和“先检查报告 / 图表 / notebook / 引用 / AI-use statement / human signoff”的最终交付 workflow。


In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/capstone_delivery_review_demo.csv')


def load_deliveries(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['slide_polish_score'] = float(row['slide_polish_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_deliveries(DATA_PATH)
print(f'Loaded {len(rows)} delivery review cards from {DATA_PATH.name}')
print('Python version:', platform.python_version())
print('Report structure:', ordered_counts(rows, 'report_structure_status'))
print('Citation status:', ordered_counts(rows, 'citation_status'))
print('Human signoff:', ordered_counts(rows, 'human_signoff_status'))
print('Reference route:', ordered_counts(rows, 'reference_route'))


In [ ]:
reference_ready_ids = sorted(
    row['delivery_id']
    for row in rows
    if row['reference_route'] == 'ready_to_submit'
)
top3_polish = sorted(
    rows,
    key=lambda row: row['slide_polish_score'],
    reverse=True,
)[:3]
baseline_ready_hits = sum(
    row['reference_route'] == 'ready_to_submit'
    for row in top3_polish
)
baseline_ready_precision = baseline_ready_hits / len(top3_polish)

print('Reference ready-to-submit deliveries:', reference_ready_ids)
print('Top-3 slide-polish deliveries:')
for row in top3_polish:
    print(
        f"  {row['delivery_id']}: polish={row['slide_polish_score']:.2f}, "
        f"route={row['reference_route']}, theme={row['theme']}"
    )
print('Baseline submit precision:', round(baseline_ready_precision, 3))


In [ ]:
def route_delivery(row):
    if row['human_signoff_status'] != 'signed':
        return 'blocked_signoff', 'human signoff or data boundary is not yet complete'
    if row['citation_status'] != 'ready':
        return 'blocked_signoff', 'citation or quote-level verification is not ready'
    if row['report_structure_status'] != 'ready':
        return 'revise_report', 'the report still needs a clearer question-data-baseline-result structure'
    if row['figure_evidence_status'] != 'ready':
        return 'revise_report', 'figures do not yet support the main report claims clearly enough'
    if row['notebook_repro_status'] != 'ready':
        return 'revise_report', 'the notebook still needs a clean rerun or reproducibility pass'
    if row['ai_use_statement_status'] != 'ready':
        return 'revise_report', 'AI-use statement is missing or not aligned with the AI Usage Log'
    if row['presentation_fit_status'] != 'ready':
        return 'presentation_rehearsal', 'materials are mostly ready, but the talk still needs timing and focus rehearsal'
    return 'ready_to_submit', 'all final delivery gates are satisfied'


routed_rows = []
for row in rows:
    route, reason = route_delivery(row)
    routed = dict(row)
    routed['route'] = route
    routed['route_reason'] = reason
    routed_rows.append(routed)

exact_match_count = sum(
    row['route'] == row['reference_route']
    for row in routed_rows
)
workflow_accuracy = exact_match_count / len(routed_rows)

print('Workflow routing decisions:')
for row in routed_rows:
    print(
        f"  {row['delivery_id']}: predicted={row['route']}, "
        f"reference={row['reference_route']}, reason={row['route_reason']}"
    )
print('Exact routing accuracy:', round(workflow_accuracy, 3))


In [ ]:
ready_queue = [row for row in routed_rows if row['route'] == 'ready_to_submit']
rehearsal_queue = [row for row in routed_rows if row['route'] == 'presentation_rehearsal']
revise_queue = [row for row in routed_rows if row['route'] == 'revise_report']
blocked_queue = [row for row in routed_rows if row['route'] == 'blocked_signoff']

ready_precision = sum(
    row['reference_route'] == 'ready_to_submit'
    for row in ready_queue
) / max(len(ready_queue), 1)
ready_recall = sum(
    row['route'] == 'ready_to_submit' and row['reference_route'] == 'ready_to_submit'
    for row in routed_rows
) / max(len(reference_ready_ids), 1)

print('Ready-to-submit queue:', [row['delivery_id'] for row in ready_queue])
print('Presentation-rehearsal queue:', [row['delivery_id'] for row in rehearsal_queue])
print('Revise-report queue:', [row['delivery_id'] for row in revise_queue])
print('Blocked-signoff queue:', [row['delivery_id'] for row in blocked_queue])
print('Structured-workflow submit precision:', round(ready_precision, 3))
print('Structured-workflow submit recall:', round(ready_recall, 3))


In [ ]:
def final_week_steps(row):
    steps = []
    if row['human_signoff_status'] != 'signed':
        steps.append('先解决最终签字、数据边界或公开提交权限，暂时不要提交最终版。')
    if row['citation_status'] != 'ready':
        steps.append('回到原文核查引用、数字和 quote/page-level 证据。')
    if row['report_structure_status'] != 'ready':
        steps.append('重排报告结构，让问题、数据、baseline、模型、结果和局限连成一条线。')
    if row['figure_evidence_status'] != 'ready':
        steps.append('为每张关键图写一句它支撑的 claim，删掉不能支撑正文的图。')
    if row['notebook_repro_status'] != 'ready':
        steps.append('从头运行 notebook，记录数据文件、环境和关键输出。')
    if row['ai_use_statement_status'] != 'ready':
        steps.append('根据实际 AI Usage Log 补一段具体、可检查的 AI-use statement。')
    if row['presentation_fit_status'] != 'ready':
        steps.append('把展示压到规定时间内，只保留 2 到 3 张核心证据图。')
    return steps


def delivery_package(row):
    return {
        'report': row['report_structure_status'],
        'figures': row['figure_evidence_status'],
        'notebook': row['notebook_repro_status'],
        'citations': row['citation_status'],
        'ai_use_statement': row['ai_use_statement_status'],
        'presentation': row['presentation_fit_status'],
        'signoff': row['human_signoff_status'],
    }


print('Final-week checklist for non-ready deliveries:')
for row in routed_rows:
    if row['route'] == 'ready_to_submit':
        continue
    print(f"\n{row['delivery_id']} -> {row['route']}")
    for step in final_week_steps(row):
        print(' -', step)

package = delivery_package(ready_queue[0])
print('\nDelivery package snapshot for the first ready project:')
for key, value in package.items():
    print(f'  {key}: {value}')


In [ ]:
delivery_assistant_prompt = '''你是我的 capstone 最终交付助手。

任务：
1. 先阅读 final delivery review table，而不是先润色 slides；
2. 依次检查 report、figures、notebook reproducibility、citations、
   AI-use statement、presentation timing 和 human signoff；
3. 把每个项目 route 到 ready_to_submit、presentation_rehearsal、
   revise_report 或 blocked_signoff；
4. 只有在 route 明确后，才给最后一周 revision checklist。

输出格式：
- Ready-to-submit projects
- Presentation-rehearsal queue
- Revise-report queue
- Blocked-signoff queue
- Final-week checklist
'''
print(delivery_assistant_prompt)


In [ ]:
workflow_checklist = [
    '先检查证据和签字边界，再润色 slides。',
    '每张图都应该对应一个明确 claim。',
    'notebook 需要能从头复跑，不能依赖隐藏状态。',
    '引用、AI-use statement 和 human signoff 不能靠展示排练补救。',
    '一次好展示通常只需要 2 到 3 张核心证据图。',
]

delivery_snapshot = {
    'dataset': DATA_PATH.name,
    'n_delivery_cards': len(rows),
    'baseline_top3_submit_precision': round(baseline_ready_precision, 3),
    'workflow_route_accuracy': round(workflow_accuracy, 3),
    'ready_to_submit': [row['delivery_id'] for row in ready_queue],
    'presentation_rehearsal': [row['delivery_id'] for row in rehearsal_queue],
    'blocked_signoff': [row['delivery_id'] for row in blocked_queue],
    'python_version': platform.python_version(),
}

print('Workflow checklist:')
for item in workflow_checklist:
    print(' -', item)

print('\nDelivery snapshot:')
for key, value in delivery_snapshot.items():
    print(f'  {key}: {value}')
